In [1]:
%pip install -q pandas sentence-transformers bertopic hdbscan umap-learn transformers torch

Note: you may need to restart the kernel to use updated packages.


## Lyrics analysis

Joins `lyrics.csv` (full lyrics) with `lyrics_lang.csv` (detected language) and runs:

1. **Language distribution by region** — which languages dominate each chart
2. **Lyrics cleaning** — strip tags, punctuation, normalise whitespace
3. **Topic modeling** — BERTopic with sentence-transformer embeddings
4. **Emotion classification** — per-song emotion scores (distilroberta)
5. **Cross-tabulations** — topic × region, emotion × region, emotion × topic

Outputs are saved to `outputs/<run_id>/`.

In [2]:
import re
import string
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

DATE_PATH = '2026/03/05'
LYRICS_DIR = Path('..') / 'data' / 'processed' / 'lyrics' / DATE_PATH

lyrics_df = pd.read_csv(LYRICS_DIR / 'lyrics.csv')
lang_df   = pd.read_csv(LYRICS_DIR / 'lyrics_lang.csv')

# Join on spotify_uri to merge language into lyrics
df = lyrics_df.merge(lang_df[['spotify_uri', 'language']], on='spotify_uri', how='left')

print(f'Loaded {len(df)} songs')
print(f'Columns: {list(df.columns)}')
df.head()

Loaded 606 songs
Columns: ['rank', 'artist', 'title', 'region', 'spotify_uri', 'lyrics', 'language']


,rank,artist,title,region,spotify_uri,lyrics,language
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,"(Dímelo, ¿me vas a dar lo que yo pido?)\nDame ...",es
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,"ARIA VEGA, Ryan Castro\nLa costeñita premium y...",es
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...",es
3,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nQué chimba, SOG\nT...",es
4,4,Kris R.,GANAS,Colombia,4KE9Ne3hgh18B3Th4xcylg,"Yeah, yeah\nYeah, yeah\n\nMi amor, culeemos co...",es


### 1) Language distribution by region

In [3]:
lang_region = pd.crosstab(df['region'], df['language'])
print(lang_region.to_string())

# Percentage view
lang_region_pct = pd.crosstab(df['region'], df['language'], normalize='index').round(3) * 100
print('\nPercentage:')
print(lang_region_pct.to_string())

language  de   en   es  gd  he  id  it  ja  ko  pt  ru  tr  unknown  vi  zh
region                                                                     
Colombia   0    8  102   0   1   0   0   0   0   0   0   0        8   0   0
Global     0  144   33   0   0   1   2   0   0   1   1   5        2   0   0
Taiwan     0   56    0   1   0   0   0   1  12   0   1   0       25   1  27
USA        1  145   18   0   0   0   2   1   0   1   0   4        2   0   0

Percentage:
language   de    en    es   gd   he   id   it   ja   ko   pt   ru   tr  unknown   vi    zh
region                                                                                    
Colombia  0.0   6.7  85.7  0.0  0.8  0.0  0.0  0.0  0.0  0.0  0.0  0.0      6.7  0.0   0.0
Global    0.0  76.2  17.5  0.0  0.0  0.5  1.1  0.0  0.0  0.5  0.5  2.6      1.1  0.0   0.0
Taiwan    0.0  45.2   0.0  0.8  0.0  0.0  0.0  0.8  9.7  0.0  0.8  0.0     20.2  0.8  21.8
USA       0.6  83.3  10.3  0.0  0.0  0.0  1.1  0.6  0.0  0.6  0.0  2.3      

### 2) Clean lyrics

In [4]:
def clean_lyrics(text: str) -> str:
    """Strip section tags, lowercase, remove punctuation, collapse whitespace."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = re.sub(r'\[[^\]]*\]', ' ', text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_lyrics'] = df['lyrics'].fillna('').map(clean_lyrics)

# Drop songs with no lyrics
has_lyrics = df['clean_lyrics'].str.strip().ne('')
print(f'Songs with lyrics: {has_lyrics.sum()} / {len(df)}')
df_valid = df[has_lyrics].reset_index(drop=True)
print(f'Proceeding with {len(df_valid)} songs')

Songs with lyrics: 569 / 606
Proceeding with 569 songs


### 3) Sentence embeddings + BERTopic

In [5]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(
    df_valid['clean_lyrics'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f'Embeddings shape: {embeddings.shape}')

/Users/wednesday/Documents/GitHub/lyrics_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8027.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 18/18 [00:07<00:00,  2.51it/s]

Embeddings shape: (569, 384)


In [6]:
n_samples = len(df_valid)
n_neighbors = max(2, min(15, n_samples - 1))
min_cluster_size = max(2, min(10, n_samples))
min_samples_val = max(1, min(5, n_samples))

umap_model = UMAP(
    n_neighbors=n_neighbors,
    n_components=2,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=min_samples_val,
    metric='euclidean',
    prediction_data=True,
)

topic_model = BERTopic(
    verbose=True,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
)

topics, probs = topic_model.fit_transform(
    df_valid['clean_lyrics'].tolist(), embeddings
)
print(f'Found {len(set(topics)) - (1 if -1 in topics else 0)} topics (+ outlier topic -1)')

2026-03-11 16:41:29,830 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 16:41:34,585 - BERTopic - Dimensionality - Completed ✓
2026-03-11 16:41:34,586 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 16:41:34,608 - BERTopic - Cluster - Completed ✓
2026-03-11 16:41:34,611 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 16:41:34,692 - BERTopic - Representation - Completed ✓


Found 30 topics (+ outlier topic -1)


In [7]:
# Topic assignments
df_valid = df_valid.copy()
df_valid['topic'] = topics
df_valid['topic_probability'] = [
    float(np.max(p)) if isinstance(p, np.ndarray) and p.size > 0 else np.nan
    for p in probs
]

# Topic keywords
topic_keywords_rows = []
for tid in sorted(t for t in set(topics) if t != -1):
    words_scores = topic_model.get_topic(tid) or []
    topic_keywords_rows.append({
        'topic': tid,
        'keywords': ', '.join(w for w, _ in words_scores[:10]),
    })
topic_keywords = pd.DataFrame(topic_keywords_rows)

print('Topic keywords:')
print(topic_keywords.to_string(index=False))

Topic keywords:
 topic                                                         keywords
     0                    the, it, of, all, you, oh, your, my, now, and
     1                          que, te, el, yo, la, de, no, pa, lo, en
     2                  zi, de, awake, wide, xing, shu, so, ah, hun, hu
     3              the, id, was, goodbye, next, it, you, wanna, to, if
     4                easy, me, fall, be, you, so, better, its, he, the
     5         can, how, when, and, we, im, bed, youve, someone, shared
     6                         la, lo, que, yo, te, en, no, de, el, por
     7    tryin, got, she, glamorous, they, it, the, that, and, highway
     8   every, you, dance, watching, ooh, take, ill, soul, darling, be
     9         2018, feat, 2017, the, lil, drake, west, 2016, of, scott
    10        bang, jennie, drip, fashion, yeah, 404, bae, rich, am, ah
    11          body, thats, what, her, like, she, on, and, lucky, baby
    12                       ey, que, voa, la, l

In [8]:
# Topic distribution by region
topic_by_region = pd.crosstab(
    df_valid['region'], df_valid['topic'], normalize='index'
).reset_index()
print('Topic distribution by region (%):')
print((topic_by_region.set_index('region') * 100).round(1).to_string())

Topic distribution by region (%):
topic       -1    0     1     2    3    4    5     6    7    8    9    10   11   12   13   14   15   16   17   18   19   20   21   22   23   24   25   26   27   28   29
region                                                                                                                                                                  
Colombia  18.0  0.9  32.4   0.0  0.0  0.0  0.0  14.4  0.0  0.0  0.0   0.0  0.9  3.6  0.0  0.9  0.0  0.0  0.0  0.9  6.3  0.0  0.0  0.0  1.8  0.0  4.5  1.8  3.6  6.3  3.6
Global     8.0  8.6   1.6   0.0  7.0  5.3  4.8   1.6  3.7  5.3  4.3   0.0  2.7  3.2  2.7  3.2  3.2  3.2  3.2  3.7  1.6  2.1  2.7  3.2  3.2  2.7  1.6  3.2  1.6  1.1  1.6
Taiwan     1.0  7.1   0.0  32.3  6.1  3.0  3.0   0.0  1.0  2.0  3.0  17.2  1.0  0.0  6.1  2.0  0.0  3.0  3.0  1.0  0.0  0.0  3.0  0.0  0.0  4.0  0.0  1.0  0.0  0.0  0.0
USA        5.2  9.3   0.0   0.6  7.6  7.0  6.4   0.0  6.4  4.1  3.5   0.0  5.2  3.5  1.7  2.9  4.7  2.9  2.9  2.3  1.7  5

### 4) Emotion classification

In [9]:
from transformers import pipeline as hf_pipeline

emotion_clf = hf_pipeline(
    'text-classification',
    model='j-hartmann/emotion-english-distilroberta-base',
    return_all_scores=True,
)
print('Emotion classifier loaded.')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26316.22it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Emotion classifier loaded.


In [10]:
def chunk_text(text: str, max_words: int = 180) -> list[str]:
    words = text.split()
    if not words:
        return ['']
    return [' '.join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

def emotion_scores(text: str) -> dict:
    chunks = chunk_text(text)
    all_scores = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        raw = emotion_clf(chunk, truncation=True)
        if isinstance(raw, list) and raw and isinstance(raw[0], list):
            raw = raw[0]
        all_scores.append(raw)
    if not all_scores:
        return {}
    labels = sorted({item['label'] for scores in all_scores for item in scores})
    averaged = {}
    for label in labels:
        vals = [next((x['score'] for x in s if x['label'] == label), 0.0) for s in all_scores]
        averaged[label] = float(np.mean(vals))
    return averaged

print(f'Computing emotion scores for {len(df_valid)} songs...')
emo_series = df_valid['clean_lyrics'].map(emotion_scores)
emo_df = pd.json_normalize(emo_series).fillna(0.0)
emo_df.columns = [f'emotion_{c}' for c in emo_df.columns]

df_valid = pd.concat([df_valid.reset_index(drop=True), emo_df], axis=1)

emotion_cols = [c for c in df_valid.columns if c.startswith('emotion_')]
if emotion_cols:
    df_valid['dominant_emotion'] = (
        df_valid[emotion_cols].idxmax(axis=1).str.replace('emotion_', '', regex=False)
    )

print('Done.')
df_valid[['title', 'region', 'language', 'dominant_emotion']].head(10)

Computing emotion scores for 569 songs...
Done.


,title,region,language,dominant_emotion
0,Las Muñequitas,Colombia,es,neutral
1,CHÉVERE (premium_remix),Colombia,es,joy
2,LA VILLA,Colombia,es,neutral
3,LA VILLA,Colombia,es,neutral
4,GANAS,Colombia,es,neutral
5,La Plena - W Sound 05,Colombia,en,joy
6,La Plena - W Sound 05,Colombia,en,joy
7,no tiene sentido,Colombia,es,neutral
8,Tonto,Colombia,es,neutral
9,quédate,Colombia,es,neutral


### 5) Cross-tabulations & save outputs

In [11]:
emotion_by_region = (
    df_valid.groupby('region')[emotion_cols].mean().reset_index()
    if emotion_cols else pd.DataFrame()
)
emotion_by_topic = (
    df_valid.groupby('topic')[emotion_cols].mean().reset_index()
    if emotion_cols else pd.DataFrame()
)
emotion_by_language = (
    df_valid.groupby('language')[emotion_cols].mean().reset_index()
    if emotion_cols else pd.DataFrame()
)

print('Emotion by region:')
print(emotion_by_region.to_string(index=False))
print('\nEmotion by topic:')
print(emotion_by_topic.to_string(index=False))
print('\nEmotion by language:')
print(emotion_by_language.to_string(index=False))

Emotion by region:
  region  emotion_anger  emotion_neutral  emotion_joy  emotion_fear  emotion_disgust  emotion_sadness  emotion_surprise
Colombia       0.026189         0.301945     0.048270      0.051681         0.007885         0.018186          0.009019
  Global       0.054750         0.106545     0.085603      0.067052         0.000000         0.221905          0.098847
  Taiwan       0.050664         0.179287     0.109355      0.045592         0.005671         0.161105          0.050961
     USA       0.072051         0.068157     0.088322      0.070707         0.001694         0.242703          0.093857

Emotion by topic:
 topic  emotion_anger  emotion_neutral  emotion_joy  emotion_fear  emotion_disgust  emotion_sadness  emotion_surprise
    -1       0.023955         0.233461     0.036903      0.115817         0.000000         0.076418          0.057561
     0       0.049864         0.000000     0.047730      0.103302         0.000000         0.362336          0.131999
     1  

In [12]:
run_id = f'lyrics_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
OUT_DIR = Path('..') / 'outputs' / run_id
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save all outputs
assignments_out = df_valid[[
    'title', 'artist', 'region', 'language', 'topic',
    'topic_probability', 'dominant_emotion'
] + emotion_cols]

assignments_out.to_csv(OUT_DIR / 'topic_assignments_per_song.csv', index=False)
topic_keywords.to_csv(OUT_DIR / 'topic_keywords.csv', index=False)
topic_by_region.to_csv(OUT_DIR / 'topic_distribution_by_region.csv', index=False)
emotion_by_region.to_csv(OUT_DIR / 'emotion_by_region.csv', index=False)
emotion_by_topic.to_csv(OUT_DIR / 'emotion_by_topic.csv', index=False)
emotion_by_language.to_csv(OUT_DIR / 'emotion_by_language.csv', index=False)

print(f'All outputs saved to {OUT_DIR}/')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')

All outputs saved to ../outputs/lyrics_20260311_164409/
  emotion_by_language.csv
  emotion_by_region.csv
  emotion_by_topic.csv
  topic_assignments_per_song.csv
  topic_distribution_by_region.csv
  topic_keywords.csv
